In [ ]:
!rm -rf /kaggle/working/Real-ESRGAN
!git clone --depth 1 --branch Kaggle https://github.com/aksjfds/Real-ESRGAN.git /kaggle/working/Real-ESRGAN
!pip install -q -r /kaggle/working/Real-ESRGAN/requirements.txt
!cd /kaggle/working/Real-ESRGAN && python -m py_compile realesrgan.py realesrgan_fast.py realesrgan_fast_entry.py enhance/*.py
!cd /kaggle/working/Real-ESRGAN && python realesrgan_fast_entry.py --help >/dev/null

In [ ]:
INPUT_VIDEO = "/kaggle/input/datasets/rustacean1/hanime2/cm_4.mp4"
OUTPUT_VIDEO = "/kaggle/working/realesrgan.mp4"

MODEL = "realesr-animevideov3"
MODEL_PATH = ""
SCALE = 2
FPS = "source"

START_TIME = 3 * 60 + 15
TEST_SECONDS = 10
PROGRESS_INTERVAL = 60.0

FP16 = True
CHANNELS_LAST = True

AUTO_TILE = True
MAX_TILE_SIZE = 1536
AUTO_BATCH = True
MAX_BATCH_SIZE = 32
TILE_SIZE = 256
TILE_PAD = 10
TILE_VERIFY_COVERAGE = False
BATCH_SIZE = 4
GPU_IDS = "0,1"

COLOR_POLICY = "preserve"
HDR_POLICY = "reject"

# Kaggle default: use the GPU encoder. libsvtav1 remains available if the installed ffmpeg advertises it.
VIDEO_CODEC = "hevc_nvenc"
OUTPUT_PIX_FMT = "auto"
CRF = 18
PRESET = "medium"
CQ = 18
NVENC_PRESET = "p7"
SVTAV1_PRESET = 6
ENCODE_GPU = 0

AUDIO_CODEC = "copy"
AUDIO_BITRATE = "192k"


In [ ]:
import shlex
import subprocess
import sys

def boolean_flag(enabled, yes, no):
    return yes if enabled else no

command = [
    sys.executable, "/kaggle/working/Real-ESRGAN/realesrgan_fast_entry.py",
    "--input", INPUT_VIDEO,
    "--output", OUTPUT_VIDEO,
    "--model", MODEL,
    "--model-path", MODEL_PATH,
    "--scale", str(SCALE),
    "--fps", str(FPS),
    boolean_flag(FP16, "--fp16", "--no-fp16"),
    boolean_flag(CHANNELS_LAST, "--channels-last", "--no-channels-last"),
    boolean_flag(AUTO_TILE, "--auto-tile", "--no-auto-tile"),
    "--max-tile-size", str(MAX_TILE_SIZE),
    boolean_flag(AUTO_BATCH, "--auto-batch", "--no-auto-batch"),
    "--max-batch-size", str(MAX_BATCH_SIZE),
    "--tile-size", str(TILE_SIZE),
    "--tile-pad", str(TILE_PAD),
    boolean_flag(TILE_VERIFY_COVERAGE, "--tile-verify-coverage", "--no-tile-verify-coverage"),
    "--batch-size", str(BATCH_SIZE),
    "--gpu-ids", GPU_IDS,
    "--color-policy", COLOR_POLICY,
    "--hdr-policy", HDR_POLICY,
    "--video-codec", VIDEO_CODEC,
    "--output-pix-fmt", OUTPUT_PIX_FMT,
    "--crf", str(CRF),
    "--preset", PRESET,
    "--cq", str(CQ),
    "--nvenc-preset", NVENC_PRESET,
    "--svtav1-preset", str(SVTAV1_PRESET),
    "--encode-gpu", str(ENCODE_GPU),
    "--audio-codec", AUDIO_CODEC,
    "--audio-bitrate", AUDIO_BITRATE,
    "--start-time", str(START_TIME),
    "--test-seconds", str(TEST_SECONDS),
    "--progress-interval", str(PROGRESS_INTERVAL),
    "--ffmpeg-bin", "ffmpeg",
    "--ffprobe-bin", "ffprobe",
]

print("[command]", shlex.join(command), flush=True)
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
assert process.stdout is not None
for line in process.stdout:
    print(line, end="", flush=True)
return_code = process.wait()
if return_code != 0:
    raise RuntimeError(f"Real-ESRGAN exited with code {return_code}; the child-process error is shown above.")
